### 2022

### HS4

### HS22

In [1]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.algorithms import compute_eci, compute_fitness, compute_extended_fitness

1) DATI

In [ ]:
Mcp_export = pd.read_parquet(
    r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\4_digits\Mcp_export_alligned\Mcp_export_alligned_HS02_Y2002.parquet')
Mcp_import = pd.read_parquet(
    r"C:\Users\vitto\Desktop\CSH RESEARCH\data\HS22_2022_4_digits\Mcp\Mcp_import\Mcp_import_2022_filtered.parquet"
     # <-- aggiusta path
)
C = pd.read_parquet(
   r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\4_digits\C\C_4_digits_HS02.parquet')
C = C.T   # prodotti sulle righe → input nelle righe

2) RUN ALGORITMI

2.1) ECI

In [ ]:
eci, pci = compute_eci(Mcp_export)

2.2) FITNESS

In [ ]:
fitness, complexity = compute_fitness(Mcp_export, max_iter = 1000, tol = 1e-6,
                    track = False)

2.3) EXTENDED FITNESS

In [ ]:
total_complexity, extended_fitness, input_complexity, intrinsic_complexity = compute_extended_fitness(
    Mcp_export,C,alpha = 0.1, max_iter = 100000, tol = 1e-6, track = False, save_delta = False, delta_window = None)

3) TABELLE

3.1) Countries

In [ ]:
countries_table = pd.DataFrame({
    "country":                   Mcp_export.index,
    "ECI":                       eci.values,
    "fitness":                   fitness.values,
    "extended_fitness":          extended_fitness.values,
   }).reset_index(drop=True)

3.2) Products

In [ ]:
products_table = pd.DataFrame({
    "product":                   Mcp_export.columns,
    "PCI":                       pci.values,
    "complexity":                complexity.values,
    "total_complexity":          total_complexity.values,
    "input_complexity":          input_complexity.values,
    "intrinsic_complexity":      intrinsic_complexity.values,
  }).reset_index(drop=True)

4) LEGENDE (per rendere le tabelle leggibili)

4.1) Countries

In [ ]:
countries_info = pd.read_csv(
    r"C:\Users\vitto\Desktop\CSH RESEARCH\data\raw\BACI\BACI_HS22"
    r"\LEGENDE\country_codes_V202601.csv",
    usecols=["country_code", "country_name"],
)
countries_table = countries_table.merge(
    countries_info, left_on="country", right_on="country_code", how="left"
)

4.2) Products

In [ ]:
products_info = pd.read_csv(
    r"C:\Users\vitto\Desktop\CSH RESEARCH\data\raw\BACI\BACI_HS22"
    r"\LEGENDE\product_codes_HS22_4_V202601.csv",
    usecols=["code", "description"],
)
products_info["code"]     = products_info["code"].astype(str).str.zfill(4)
products_table["product"] = products_table["product"].astype(str)

products_table = products_table.merge(
    products_info, left_on="product", right_on="code", how="left"
)

5) SALVATAGGIO

In [ ]:
countries_table.to_parquet(r'C:\Users\vitto\Desktop\CSH RESEARCH\results\real_data\countries_table.parquet')
products_table.to_parquet(r'C:\Users\vitto\Desktop\CSH RESEARCH\results\real_data\products_table.parquet')